# Chute Template & Sandbox Manual Tests

This notebook exercises the Chute builder, sandbox training entrypoint, and miner runtime flows that were highlighted for manual verification.

## 1. Bootstrap (Optional)

Run this cell if you have not already cloned the repository and installed dependencies.

In [ ]:
!git clone https://github.com/tensorlink-dev/epochor.git
%cd epochor
!pip install -r requirements.txt
!pip install bittensor fastapi uvicorn apscheduler safetensors datasets

## 2. Chute Builder Smoke Test

Build the default validator training Chute and inspect the resulting configuration.

In [ ]:
from templates.validator_training.chute import build_validator_training_template

chute = build_validator_training_template("your-handle")
print(chute.image.name)
print(chute.node_selector.gpu_count, chute.node_selector.min_vram_gb_per_gpu)
print(chute.run.entry_file, chute.run.entry_point)
print(chute.environment)

## 3. Chute Overrides

Apply common overrides (GPU count, VRAM, concurrency, timeout, max instances, extra pip packages) and ensure they are reflected in the configuration.

In [ ]:
override_chute = build_validator_training_template(
    "your-handle",
    gpu_count=2,
    min_vram_gb_per_gpu=24,
    concurrency=2,
    timeout_seconds=7200,
    max_instances=4,
    extra_pip=["wandb", "accelerate"]
)
print(override_chute.node_selector.gpu_count, override_chute.node_selector.min_vram_gb_per_gpu)
print(override_chute.concurrency, override_chute.timeout_seconds, override_chute.max_instances)
print(override_chute.environment["EXTRA_PIP_PACKAGES"])

## 4. Submission Loader Validation

Test the sandbox loader against submissions that expose either a `Submission` class or `get_submission` function, and confirm type errors surface for invalid exports.

In [ ]:
import tempfile, textwrap, os
from templates.validator_training.trainer_entry import _load_submission
from templates.validator_training import MinerSubmissionProtocol

root_dir = tempfile.mkdtemp()
valid_submission_dir = os.path.join(root_dir, 'valid_submission')
os.makedirs(valid_submission_dir, exist_ok=True)
with open(os.path.join(valid_submission_dir, 'miner.py'), 'w') as fh:
    fh.write(textwrap.dedent('''from templates.validator_training import MinerSubmissionProtocol

class Submission(MinerSubmissionProtocol):
    def build_model(self, cfg):
        import torch.nn as nn
        return nn.Linear(1, 1)
    def build_optimizer(self, model, cfg):
        import torch.optim as optim
        return optim.SGD(model.parameters(), lr=0.1)
    def train_step(self, model, batch, optimizer, step_idx, cfg):
        return {"loss": 0.0}
'''))

submission = _load_submission(valid_submission_dir)
print(isinstance(submission, MinerSubmissionProtocol))

factory_dir = os.path.join(root_dir, 'factory_submission')
os.makedirs(factory_dir, exist_ok=True)
with open(os.path.join(factory_dir, 'miner.py'), 'w') as fh:
    fh.write(textwrap.dedent('''from templates.validator_training import MinerSubmissionProtocol

class SubmissionImpl(MinerSubmissionProtocol):
    def build_model(self, cfg):
        import torch.nn as nn
        return nn.Linear(1, 1)
    def build_optimizer(self, model, cfg):
        import torch.optim as optim
        return optim.SGD(model.parameters(), lr=0.1)
    def train_step(self, model, batch, optimizer, step_idx, cfg):
        return {"loss": 0.0}


def get_submission():
    return SubmissionImpl()
'''))

submission_factory = _load_submission(factory_dir)
print(isinstance(submission_factory, MinerSubmissionProtocol))

error_dir = os.path.join(root_dir, 'invalid_submission')
os.makedirs(error_dir, exist_ok=True)
with open(os.path.join(error_dir, 'miner.py'), 'w') as fh:
    fh.write('value = 5')

try:
    _load_submission(error_dir)
except TypeError as exc:
    print('Raised TypeError as expected:', exc)

## 5. Determinism & Network Guardrails

Run the sandbox entrypoint twice with identical seeds and network blocking to verify deterministic metrics and socket restrictions.

In [ ]:
from templates.validator_training import trainer_entry

config = {
    'seed': 123,
    'max_steps': 3,
    'disable_network': True,
    'train_batch_size': 8,
}

first_run = trainer_entry.run(submission_dir=valid_submission_dir, cfg=config, artifacts_dir=None)
second_run = trainer_entry.run(submission_dir=valid_submission_dir, cfg=config, artifacts_dir=None)
print(first_run['metrics'], second_run['metrics'])

## 6. Toy Dataloader Sanity

Inspect the shapes emitted by the toy dataset helper across different configuration options.

In [ ]:
from templates.validator_training.trainer_entry import _ToyDataset, _get_dataloader

dataset = _ToyDataset(length=64, input_dim=4, seed=7)
dataloader = _get_dataloader(dataset, batch_size=16)
for batch_idx, batch in enumerate(dataloader):
    print(batch_idx, batch['inputs'].shape, batch['targets'].shape)
    if batch_idx == 2:
        break

## 7. Checkpoint Resume & Artifact Hashing

Trigger a run that emits a checkpoint and then resume from it to validate state restoration and SHA-256 hashing.

In [ ]:
import pathlib
resume_cfg = {'seed': 321, 'max_steps': 2, 'train_batch_size': 4}
outcome = trainer_entry.run(submission_dir=valid_submission_dir, cfg=resume_cfg, artifacts_dir=None)
ckpt_path = pathlib.Path(outcome['artifacts']['checkpoints'][0]['path'])
print(ckpt_path.exists(), outcome['artifacts']['checkpoints'][0]['sha256'])
resume_cfg['resume_ckpt_path'] = str(ckpt_path)
resume_outcome = trainer_entry.run(submission_dir=valid_submission_dir, cfg=resume_cfg, artifacts_dir=None)
print(resume_outcome['metrics'])

## 8. Hugging Face Window Stream Smoke Test
Instantiate the deterministic window streamer with a small budget, consume a few batches, and confirm the tensors have the expected shapes. This exercises the new `hf_window_stream` settings and proves that the trainer can read from Hugging Face without exhausting the iterator.


In [ ]:
import os
from templates.validator_training import trainer_entry

hf_cfg = {
    'hf_dataset_repo': 'tensorlink-dev/gifteval-iid',
    'hf_dataset_split': 'train',
    'hf_target_key': 'target',
    'hf_window_stream': {
        'context_length': 64,
        'forecast_horizon': 16,
        'max_batches': 4,
        'budget_batch_size': 8,
        'total_shards': 64,
        'active_shards': 2,
        'streams_per_shard': 2,
    },
}

dataset = trainer_entry._build_dataset(hf_cfg, submission_dir=os.getcwd())
loader = trainer_entry._get_dataloader({'train_batch_size': 8}, dataset)
first_batch = next(loader)
first_batch['x'].shape, first_batch['y'].shape


## 9. Hugging Face Upload Paths

Simulate successful and failing uploads by toggling credentials.

In [ ]:
from templates.validator_training.trainer_entry import _prepare_upload_payload

payload = _prepare_upload_payload(outcome['artifacts'])
print(payload.keys())

config_no_token = resume_cfg | {'hf_write_token_env': 'HF_WRITE_TOKEN_ENV'}
try:
    trainer_entry.run(submission_dir=valid_submission_dir, cfg=config_no_token, artifacts_dir=None)
except Exception as exc:
    print('Upload failure surfaced:', type(exc).__name__)

## 10. Miner CLI Smoke Test

Inspect the miner CLI flags and run it in dry mode to ensure registration logic initializes without crashing.

In [ ]:
!python neurons/miner.py --help

In [ ]:
!python neurons/miner.py --netuid 1 --subtensor.network finney --wallet.name test --wallet.hotkey test --offline